# Swarm obstacle avoidance: long RL training on a Kaggle GPU

Same learning problem as `scripts/rl_train.py` on the laptop (PPO, one shared policy for every follower),
but the simulator runs batched on the GPU (`swarm_tools/torch_sim.py`, checked step for step against the
laptop simulator) with 1024 missions at a time.

**Before you start** (full steps, and what to do with the results: `docs/KAGGLE_GUIDE.md`):
1. *Add Input* → your dataset **swarm-rl-code** (the zip `kaggle/swarm-rl-code.zip` from the laptop).
2. *Settings* → *Accelerator*: **GPU T4 x2**. Internet is not needed.
3. For a second seed, change `SEED` below and run the notebook again.
4. Start it with **Save Version → Save & Run All (Commit)**, so it runs in the background (about 11 h).

The best policy is chosen on 90 *validation* courses; the final results use the 90 *test* courses of the laptop evaluation, which training and selection never see.

Everything the laptop needs afterwards is packed into `/kaggle/working/results.zip` (Output tab).

In [ ]:
# Cell 1: settings, GPU check, unpack the code
SEED = 1            # change for more seeds (2, 3, ...)
HOURS = 10.5        # Kaggle stops a session at 12 h; the trainer saves and stops cleanly before that
STEPS = 6e9         # training target; on a T4 (~140k steps/s) the 10.5 h limit comes first
import glob, os, shutil, subprocess, zipfile, torch
print("torch", torch.__version__, "CUDA", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
src = glob.glob('/kaggle/input/**/swarm-rl-code.zip', recursive=True)
os.makedirs('/kaggle/working/code', exist_ok=True)
if src:
    zipfile.ZipFile(src[0]).extractall('/kaggle/working/code')
else:  # Kaggle unpacked the zip already
    top = glob.glob('/kaggle/input/**/scripts/rl_train_gpu.py', recursive=True)[0]
    shutil.copytree(os.path.dirname(os.path.dirname(top)), '/kaggle/working/code', dirs_exist_ok=True)
os.chdir('/kaggle/working/code')
ENV = dict(os.environ, PYTHONPATH='src')
print(sorted(os.listdir('.')))

In [ ]:
# Cell 2: scenario pools (~10 min on 4 CPUs): 8000 training courses, 90 validation courses (pick the best policy)
#    and the 90 test courses of the laptop evaluation (only for the final report)
subprocess.run(['python', 'scripts/build_pools.py', '--train', '8000', '--workers', '4', '--out', '/kaggle/working/pools'],
               env=ENV, check=True)

In [ ]:
# Cell 3: training on the GPU. Resumes automatically if /kaggle/working/run_seed{SEED}/ckpt.pt exists
#    (to continue an earlier session, copy that session's run folder there first - see docs/KAGGLE_GUIDE.md).
run = f'/kaggle/working/run_seed{SEED}'
subprocess.run(['python', 'scripts/rl_train_gpu.py', '--pool', '/kaggle/working/pools/train.npz',
                '--eval-pool', '/kaggle/working/pools/val.npz', '--run', run, '--seed', str(SEED),
                '--hours', str(HOURS), '--steps', str(STEPS), '--envs', '1024'], env=ENV, check=True)
print(open(f'{run}/status.json').read())

In [ ]:
# Cell 4: final test with the laptop protocol (numpy simulator, CPU) on the 90 test courses, and the real-map route
pol = f'{run}/best_policy.npz'
subprocess.run(['python', 'scripts/rl_eval.py', '--policy', pol, '--episodes', '30', '--jobs', '4',
                '--out', f'/kaggle/working/eval_seed{SEED}'], env=ENV, check=True)
print(open(f'/kaggle/working/eval_seed{SEED}/summary.md').read())
subprocess.run(['python', 'scripts/rl_eval_route.py', '--policy', pol, '--seeds', '5', '--jobs', '4',
                '--out', f'/kaggle/working/route_eval_seed{SEED}'], env=ENV, check=True)
print(open(f'/kaggle/working/route_eval_seed{SEED}/summary.md').read())

In [ ]:
# Cell 5: everything to download, in one file (Output tab -> results.zip)
keep = [f'run_seed{SEED}', f'eval_seed{SEED}', f'route_eval_seed{SEED}']
with zipfile.ZipFile('/kaggle/working/results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for k in keep:
        for f in glob.glob(f'/kaggle/working/{k}/**/*', recursive=True):
            if os.path.isfile(f):
                z.write(f, os.path.relpath(f, '/kaggle/working'))
print('download /kaggle/working/results.zip', round(os.path.getsize('/kaggle/working/results.zip') / 1e6, 1), 'MB')